# Nemotron-3-Nano-30B — v0.9 SFT Training on DGX Spark GB10

**Approach:** Format 4 SFT with per-expert MoE LoRA — 878M trainable params, 11,962 PEFT keys (186 base + 11,776 per-routed-expert)  
**Hardware:** NVIDIA DGX Spark GB10 — 130.7 GB HBM, Blackwell, aarch64  
**Key advantage vs Kaggle RTX Pro 6000:** Full expert LoRA submission — the 856M routed-expert params actually load at Kaggle inference (0.58 vs 0.53–0.56 plateau)

## Hardware comparison

| | Kaggle RTX Pro 6000 | DGX Spark GB10 |
|---|---|---|
| VRAM | 96 GB | 130.7 GB HBM |
| Arch | Blackwell, x86_64 | Blackwell, **aarch64** |
| Time limit | **9 hours** | Unlimited |
| Expert LoRA | ✗ fused (Unsloth format, dropped at inference) | **✓ PEFT format, 11,776 keys loaded** |
| Trainable params | 27M (attn only, expert params discarded) | **878M** |
| Seq length | 2048–7680 | 2048–4096+ |
| Data | `v0.9_train.jsonl` 13,730 rows | `v0.12_train.jsonl` 25,500 rows |

## Prerequisites

1. Clone the project repo to the DGX Spark host
2. Set `.env` in the project root:
   ```
   HF_TOKEN=hf_...
   KAGGLE_USERNAME=gdataranger
   KAGGLE_KEY=...
   ```
3. Download the base model once:
   ```zsh
   huggingface-cli download nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 \
     --local-dir .cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
   ```
4. Build the Docker image (see next section)
5. Training data: download `gdataranger/nemotron-v09-training-data` from Kaggle → `data/v0.9_train.jsonl`

## Run history

| Run | Script | Steps | Seq | Warmstart | Expert LoRA keys | Kaggle Score |
|---|---|---|---|---|---|---|
| run12 | `run_train_v9.sh` | 330 | 2048 | None | 0 (packaging bug) | ~0.55 |
| **run13** | `run_train_v9.sh` | **1000** | 2048 | None | **11,962** | **0.58** ★ (step 500) |
| run14 | `run_train_v12.sh` | 600 | 4096 | run13 step-1000 | 11,962 | active |


## Dockerfile.gb10 — Training image for DGX Spark

This is the exact Dockerfile used to build `nemotron-gb10:latest` on the DGX Spark. Key differences from the Kaggle build script:

- Base: `nvcr.io/nvidia/pytorch:26.05-py3` (aarch64, SM 12.0)
- `causal-conv1d` and `mamba-ssm` built from source with `TORCH_CUDA_ARCH_LIST=...12.0;12.1+PTX`
- `peft==0.19.1` (enables `target_parameters` for per-expert batched LoRA — Kaggle notebooks have 0.14.0)
- `bitsandbytes` built from source for SM 12.0
- No GPU inside `docker build` — `selective_scan_cuda` import is patched to be optional

```Dockerfile
# Nemotron LoRA training image — NVIDIA PyTorch 26.05 rebase for GB10 / DGX Spark.
#
# Key differences from 26.04:
#   - Base image: nvcr.io/nvidia/pytorch:26.05-py3  (was 26.04)
#   - Container CUDA toolkit: 13.2.1 (same as 26.04 — both use CUDA 13.2.1.009)
#   - torch.version.cuda: likely still "12.1" — PyTorch in NGC containers is compiled
#     against CUDA 12.1 by design for broad GPU compatibility. The toolkit version and
#     the PyTorch build CUDA are deliberately decoupled. Our JIT-compiled extensions
#     (causal-conv1d, mamba-ssm, bitsandbytes) use nvcc 13.2 with SM 12.0 targets and
#     are unaffected. PyTorch built-in kernels fall back to PTX on SM 12.0 (correct,
#     slightly slower). Watch NGC release notes for when torch.version.cuda bumps to 13.x.
#   - Verify Python version after pull (26.05 may ship 3.13 instead of 3.12 — update
#     lib path strings in the RUN python3 patch steps below if so)
#   - All Python package pins unchanged from 26.04 until verified
#
# Known version issues (carried from 26.04, monitor for fixes in 26.05):
#   - peft==0.19.1: upgraded from 0.14.0 to enable target_parameters support (peft
#     0.19+). This is required for per-expert batched LoRA on NemotronH MoE layers —
#     peft 0.14.0 silently ignores LoraConfig.target_parameters, which caused
#     trainable params to stay at 27.7M (no routed-expert LoRA) instead of ~455M.
#     Unsloth 2026.6.2 was designed for peft >= 0.19 and is compatible.
#     Kaggle adapters saved with peft_version="0.18.1" load fine; UserWarning about
#     'peft_version' key is cosmetic and harmless.
#   - NemotronHForCausalLM.supports_gradient_checkpointing=False: standard
#     gradient_checkpointing_enable() raises ValueError. Worked around in
#     train_v9_sft.py by no-op override after native _set_gradient_checkpointing().
#     May be fixed in newer transformers — check after upgrade.
#   - Unsloth 2026.6.2 does not patch gradient_checkpointing_enable() for NemotronH
#     on the DGX build (Kaggle build script version does). Revisit if Unsloth updates.
#   - torchvision version mismatch (cosmetic): Unsloth warns at import but training
#     is unaffected. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence.

FROM --platform=linux/arm64 nvcr.io/nvidia/pytorch:26.05-py3

ENV DEBIAN_FRONTEND=noninteractive \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1 \
    HF_HOME=/workspace/.cache/huggingface \
    HUGGINGFACE_HUB_CACHE=/workspace/.cache/huggingface \
    TOKENIZERS_PARALLELISM=false \
    BASE_MODEL_ID=nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 \
    ADAPTER_OUTPUT_DIR=/workspace/output/adapter \
    SUBMISSION_DIR=/workspace/output/submission

WORKDIR /workspace

# Print base environment for build log validation.
RUN python3 -c "import torch, sys; \
    print('Python:', sys.version); \
    print('PyTorch:', torch.__version__); \
    import subprocess; \
    v = subprocess.run(['nvcc','--version'], capture_output=True, text=True).stdout.strip().splitlines()[-1]; \
    print('CUDA toolkit:', v)"

# Show active pip constraints so any conflicts with our pins are visible in the build log.
RUN echo '=== /etc/pip/constraint.txt ===' \
 && cat /etc/pip/constraint.txt || echo '(not present)'

RUN apt-get update && apt-get install -y --no-install-recommends \
    git git-lfs wget curl vim nano build-essential cmake pkg-config unzip zip \
 && rm -rf /var/lib/apt/lists/*

RUN git lfs install
RUN python -m pip install --upgrade pip setuptools wheel packaging
RUN python -m pip uninstall -y apex || true

RUN pip install \
    "transformers==5.5.3" \
    "datasets==3.2.0" \
    "accelerate==1.3.0" \
    "peft==0.19.1" \
    "trl==0.15.2" \
    "torchao==0.17.0" \
    "huggingface_hub>=1.5.0,<2.0" \
    "sentencepiece" \
    "safetensors" \
    "scipy" \
    "evaluate" \
    "scikit-learn" \
    "pydantic" \
    "pandas" \
    "tqdm" \
    "jsonschema" \
    "dspy-ai>=2.4.0" \
    "fastapi" \
    "uvicorn[standard]" \
    "pyyaml" \
    "gradio" \
    "neo4j" \
    "kagglehub"

# causal-conv1d: source build required for SM 12.0 (Blackwell/GB10) support.
# TORCH_CUDA_ARCH_LIST targets all arches from Ampere through Blackwell (≥ 8.0).
#
# mamba-ssm / selective_scan_cuda: Docker RUN steps run in OCI worker containers with
# isolated device namespaces — GPU devices from the host are never visible, so
# torch.cuda.is_available() is False and mamba-ssm's setup.py silently skips building
# selective_scan_cuda. The extension is absent from the installed package, which causes
# `import selective_scan_cuda` in selective_scan_interface.py to raise ModuleNotFoundError
# at runtime.
#
# Fix: the && python3 -c "..." line below patches selective_scan_interface.py to wrap that
# import in try/except, tolerating selective_scan_cuda = None. This is safe: Nemotron-H
# uses Mamba-2 Triton kernels for its forward pass and never calls the legacy Mamba-1
# selective_scan_cuda path. The patch matches only a bare top-level import so it is a
# no-op if mamba-ssm already wraps it (as ≥ 2.3.1 does), preventing double-wrapping.
#
# Pass --build-arg MAMBA_REBUILD=$(date +%s) to force recompile on arch change.
ARG MAMBA_REBUILD=1
RUN CAUSAL_CONV1D_FORCE_BUILD=TRUE \
    TORCH_CUDA_ARCH_LIST="8.0;8.6;8.7;8.9;9.0;12.0;12.1+PTX" \
    CUDA_HOME=/usr/local/cuda \
    MAX_JOBS=8 \
    pip install causal-conv1d --no-binary causal-conv1d --no-build-isolation

RUN TORCH_CUDA_ARCH_LIST="8.0;8.6;8.7;8.9;9.0;12.0;12.1+PTX" \
    CUDA_HOME=/usr/local/cuda \
    MAX_JOBS=8 \
    pip install mamba-ssm --no-binary mamba-ssm --no-build-isolation \
 && python3 -c "f='/usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py'; c=open(f).read(); needle='\nimport selective_scan_cuda\n'; open(f,'w').write(c.replace(needle,'\ntry:\n    import selective_scan_cuda\nexcept ImportError:\n    selective_scan_cuda = None\n',1)) if needle in c else None"

# mergekit: trl==0.15.2 GRPOTrainer has a hard `import mergekit` in grpo_trainer.py,
# and mergekit itself pulls transitive deps (immutables, etc.) that conflict with our
# pinned stack. Same fix as mamba-ssm: guard the import in grpo_trainer.py with
# try/except, then ensure no broken partial install remains.
RUN pip uninstall -y mergekit llm-blender 2>/dev/null || true \
 && pip install --force-reinstall --no-deps "trl==0.15.2"

RUN python3 - <<'PYEOF'
import pathlib, re
f = pathlib.Path('/usr/local/lib/python3.12/dist-packages/trl/trainer/grpo_trainer.py')
if f.exists():
    c = f.read_text()
    # Guard both `import mergekit` and `from mergekit import ...` forms.
    c2 = re.sub(r'^(import mergekit\b.*)',
                'try:\n    \\1\nexcept ImportError:\n    mergekit = None',
                c, flags=re.MULTILINE)
    c2 = re.sub(r'^(from mergekit\b.*)',
                'try:\n    \\1\nexcept ImportError:\n    pass',
                c2, flags=re.MULTILINE)
    if c2 != c:
        f.write_text(c2)
        print('grpo_trainer.py: guarded mergekit imports OK')
    else:
        print('grpo_trainer.py: no mergekit imports found (already patched or changed)')
else:
    print('grpo_trainer.py not found at expected path')
PYEOF

# TRL 0.15.2: disable vllm (pydantic/torch conflict in this image).
# GRPOTrainer smoke test omitted — not needed for SFT training and fails without
# mergekit's full transitive dep tree installed.
RUN python3 - <<'PYEOF'
import pathlib

utils = pathlib.Path('/usr/local/lib/python3.12/dist-packages/trl/import_utils.py')
c = utils.read_text()
old = '_vllm_available = _is_package_available("vllm")'
new = '_vllm_available = False  # disabled: pydantic/torch conflict in this image'
if old in c:
    utils.write_text(c.replace(old, new, 1))
    print('import_utils.py patched OK')
else:
    print('import_utils.py: needle not found (already patched or changed)')
PYEOF

# Unsloth — FastLanguageModel patches Nemotron-H's MoE expert tensors (batched
# 3-D torch.Parameters) and Mamba projections as trainable nn.Linear-like LoRA
# targets. Without this, PeftModel.from_pretrained(v27) silently drops ~232 of
# v27's 418 adapter keys (mixer.experts.w1/w2/w3, gate_proj, x_proj), and those
# layers train on base weights only → Kaggle score ~0.56 instead of ~0.87.
# See docs/investigate/v0.5-unsloth-peft-key-mismatch.md for full root cause.
#
# Install strategy:
#   --no-deps: prevents Unsloth from downgrading our pinned transformers==5.5.3,
#   peft==0.14.0, trl==0.15.2, accelerate==1.3.0. Unsloth's dep range is
#   compatible (its lower bounds are below our pins) but pip would still
#   re-resolve and potentially downgrade. We skip compiled CUDA extras entirely
#   (no [cu121], no [cu128]) — we need only FastLanguageModel's Python-level
#   model patching, not the triton/flash-attn optimized kernels.
#
#   Re-pin after: force-reinstall critical packages to guarantee consistency
#   even if a transitive dep tried to change them.
#
#   Smoke test: verify FastLanguageModel imports cleanly before continuing.
RUN pip install unsloth unsloth_zoo --no-deps && \
    pip install \
        "transformers==5.5.3" \
        "peft==0.19.1" \
        "trl==0.15.2" \
        "accelerate==1.3.0" \
        --force-reinstall --no-deps && \
    pip show unsloth unsloth_zoo | grep -E "^Name|^Version"
# Note: `from unsloth import FastLanguageModel` cannot be tested here — unsloth_zoo
# runs a GPU presence check at import time (device_type.py:get_device_type) which
# raises NotImplementedError in GPU-less Docker build containers. This is the same
# pattern as mamba-ssm and causal-conv1d. The import is guarded by try/except in
# train_v5_sft.py and will succeed at runtime when --privileged + NVIDIA_VISIBLE_DEVICES
# are set.

RUN pip install scikit-build-core
# Build bitsandbytes to /opt/bitsandbytes (not /workspace) so the -v /workspace mount
# at runtime doesn't hide the source. Use non-editable install so compiled .so files
# are copied to site-packages and survive without the source directory at runtime.
RUN git clone https://github.com/bitsandbytes-foundation/bitsandbytes.git /opt/bitsandbytes \
 && cd /opt/bitsandbytes \
 && cmake -DCOMPUTE_BACKEND=cuda -DCOMPUTE_CAPABILITY="80;86;87;89;90;120;121" -S . -B build \
 && cmake --build build --config Release -j8 \
 && pip install . --no-build-isolation

RUN mkdir -p \
    /workspace/data \
    /workspace/configs \
    /workspace/scripts \
    /workspace/output \
    /workspace/.cache/huggingface

COPY *.py /workspace/
COPY data /workspace/data
COPY configs /workspace/configs
COPY scripts /workspace/scripts

CMD ["/bin/bash"]

```

## Build the Docker image

Run once on the DGX Spark host. Never import an x86_64 image — `causal_conv1d` and `mamba_ssm` must be compiled for `aarch64 + SM 12.0`.

```zsh
# From project root (~20 min first build, cached thereafter)
docker build \
  --platform linux/arm64 \
  -f Dockerfile.gb10 \
  -t nemotron-gb10:latest \
  .
```

Verify the image:

```zsh
docker run --rm --privileged \
  -e NVIDIA_VISIBLE_DEVICES=all \
  nemotron-gb10:latest \
  python -c "
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info(0)
print(f'Free: {free/1e9:.1f} GB / {total/1e9:.1f} GB')
"
```

In [ ]:
import os

# ── RUN IDENTITY ──────────────────────────────────────────────────────────
RUN_NAME   = "v9_run13"                 # matches run in leaderboard.md
OUTPUT_DIR = f"output/adapter_{RUN_NAME}"

# ── INCREMENTAL TRAINING ──────────────────────────────────────────────────
# None = fresh start; set to prior adapter path for warmstart
WARMSTART_ADAPTER = None              # run13: fresh; run14: "output/adapter_v9_run13_ckpt"

# ── DATA ─────────────────────────────────────────────────────────────────
TRAIN_FILE     = "data/v0.9_train.jsonl"   # run14: "data/v0.12_train.jsonl"
MIN_SEQ_LENGTH = 0
MAX_SEQ_LENGTH = 2048     # run13: 2048; run14: 4096 (8192 OOM risk at 878M params)

# ── TRAINING HYPERPARAMETERS ─────────────────────────────────────────────
MAX_STEPS     = 1000      # run13: 1000 (~25h at 90s/step); run14: 600 (~20h at 122s/step)
LEARNING_RATE = 2e-4      # run13: 2e-4 (fresh); run14: 1e-4 (warmstart)
BATCH_SIZE    = 1
GRAD_ACCUM    = 16        # effective batch = 16
LORA_R        = 32
LORA_ALPHA    = 32
CKPT_EVERY    = 100       # save rolling checkpoint every N steps
SEED          = 3407

print(f"RUN_NAME:          {RUN_NAME}")
print(f"OUTPUT_DIR:        {OUTPUT_DIR}")
print(f"WARMSTART_ADAPTER: {WARMSTART_ADAPTER}")
print(f"TRAIN_FILE:        {TRAIN_FILE}")
print(f"MIN_SEQ_LENGTH:    {MIN_SEQ_LENGTH}")
print(f"MAX_SEQ_LENGTH:    {MAX_SEQ_LENGTH}")
print(f"MAX_STEPS:         {MAX_STEPS}")
print(f"LEARNING_RATE:     {LEARNING_RATE}")
print(f"CKPT_EVERY:        {CKPT_EVERY}")

In [ ]:
import subprocess, shutil

if not shutil.which("docker"):
    print("Docker not available — this cell runs on the DGX Spark host only.")
    print("On Kaggle: GPU check is not applicable (training does not run here).")
else:
    result = subprocess.run(
        ["docker", "run", "--rm", "--privileged",
         "-e", "NVIDIA_VISIBLE_DEVICES=all",
         "nemotron-gb10:latest",
         "python", "-c", """
import torch
torch.cuda.init()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
used = total - free
print(f'GPU free={free/1e9:.1f}GB  total={total/1e9:.1f}GB  used={used/1e9:.1f}GB')
if free < 70e9:
    print(f'PREFLIGHT_FAIL: only {free/1e9:.1f} GB free')
elif free < 90e9:
    print(f'WARNING: {free/1e9:.1f} GB free — stale allocs present')
else:
    print('PREFLIGHT_OK: GPU memory clear')
"""],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print('stderr:', result.stderr[:500])

## Training — v0.9 run13 (`run_train_v9.sh`)

The run script wraps Docker invocation with GPU preflight, page-cache management, service pause/resume, and periodic checkpointing. Always run inside tmux.

```zsh
# Open a new tmux session first
tmux new -s train_v9

# run13: 1000 steps, seq=2048, fresh start, 878M trainable params
RUN_NAME=v9_run13 \
TRAIN_FILE=data/v0.9_train.jsonl \
MAX_SEQ_LENGTH=2048 \
MAX_STEPS=1000 \
LEARNING_RATE=2e-4 \
bash scripts/run_train_v9.sh
```

**Verify warmstart engaged** (relevant when `WARMSTART_ADAPTER` is set):
```
[moe-lora] Warmstart: loaded 92 expert LoRA weights from output/adapter_v9_run13_ckpt/expert_lora_weights.pt
```

**Verify expert LoRA injection at startup:**
```
[moe-lora] Injected per-expert LoRA into 23 × 128 experts: 11776 new PEFT keys
Trainable: 878,880,768 / 32,459,714,560 (2.71%)
```

## Training — v0.12 run14 (`run_train_v12.sh`)

v0.12 warmstarts from run13 step-1000 and trains on the 25,500-row augmented dataset.

```zsh
# Open a new tmux session first
tmux new -s train_v12

# run14: 600 steps, seq=4096, warmstart from run13, lr=1e-4
RUN_NAME=v12_spark \
TRAIN_FILE=data/v0.12_train.jsonl \
WARMSTART_ADAPTER=output/adapter_v9_run13_ckpt \
MAX_SEQ_LENGTH=4096 \
MAX_STEPS=600 \
LEARNING_RATE=1e-4 \
bash scripts/run_train_v12.sh
```

**Workspace mount**: the project directory is bind-mounted as `/workspace` inside the container.  
All relative paths (`output/`, `data/`, `scripts/`) resolve to `/workspace/<path>` inside Docker.  
Checkpoints written by the container land directly in the local `output/` directory — no SSH or copy needed.

**Data after seq=4096 filtering**: 15,502 of 25,500 examples used (9,998 dropped >4096 tokens).  
**Step time**: ~122s/step → 600 steps ≈ 20h total.

## `scripts/run_train_v9.sh` — full source

The Docker runner handles GPU preflight, page-cache management, OOM protection, and service pause/resume. Key design decisions:

- `--privileged -e NVIDIA_VISIBLE_DEVICES=all`: required for GPU on GB10 (not `--gpus all --runtime=nvidia` — see `.clinerules/14`)
- `--oom-score-adj -300`: protects training container from OOM killer
- Page-cache dropper loop (every 30s): prevents accumulated file cache from triggering OOM during backward peaks
- `ionice -c 2 -n 7`: lower I/O priority to avoid disk contention with dropper loop

```zsh
#!/usr/bin/env bash
# Run train_v9_sft.py inside the nemotron-gb10 container.
# Always run inside tmux — never run directly.
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
WORKSPACE="$(dirname "$SCRIPT_DIR")"
[[ -f "${WORKSPACE}/.env" ]] && { set -a; source "${WORKSPACE}/.env"; set +a; }

RUN_NAME="${RUN_NAME:-v9_$(date +%Y%m%d_%H%M%S)}"
TRAIN_FILE="${TRAIN_FILE:-/workspace/data/v0.9_train.jsonl}"
WARMSTART_ADAPTER="${WARMSTART_ADAPTER:-}"
MAX_SEQ_LENGTH="${MAX_SEQ_LENGTH:-2048}"
MAX_STEPS="${MAX_STEPS:-1000}"
LEARNING_RATE="${LEARNING_RATE:-2e-4}"
ADAPTER_OUT="/workspace/output/adapter_${RUN_NAME}"
LOG_FILE="${WORKSPACE}/output/train_${RUN_NAME}.log"

# ── GPU pre-flight + page-cache drop ──────────────────────────────────────
# (see scripts/run_train_v9.sh for full preflight logic)

# ── run training ──────────────────────────────────────────────────────────
ionice -c 2 -n 7 docker run --privileged \
  --name "nemotron-trainer-v9" \
  --oom-score-adj -300 \
  -e NVIDIA_VISIBLE_DEVICES=all \
  --ipc=host \
  --ulimit memlock=-1 \
  -e HF_TOKEN="${HF_TOKEN}" \
  -e HF_HUB_OFFLINE=1 \
  -e PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True,max_split_size_mb:512" \
  -v "${WORKSPACE}":/workspace \
  -v "${WORKSPACE}/.cache/huggingface":/home/ubuntu/.cache/huggingface \
  -v "${WORKSPACE}/.cache/triton":/home/ubuntu/.triton \
  -w /workspace \
  nemotron-gb10:latest \
  python scripts/train_v9_sft.py \
    --train-file     "${TRAIN_FILE}" \
    --output-dir     "${ADAPTER_OUT}" \
    --model-id       nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 \
    --max-steps      "${MAX_STEPS}" \
    --learning-rate  "${LEARNING_RATE}" \
    --max-seq-length "${MAX_SEQ_LENGTH}" \
    --batch-size     1 \
    --grad-accum     16 \
    --lora-r         32 \
    --lora-alpha     32 \
    --seed           3407 \
    --ckpt-every     100 \
    ${WARMSTART_ADAPTER:+--warmstart-adapter "${WARMSTART_ADAPTER}"} \
  2>&1 | tee "${LOG_FILE}"
```

In [ ]:
import re, os, pathlib

log_path = pathlib.Path(f"output/train_{RUN_NAME}.log")

if not log_path.exists():
    print(f"Log not found: {log_path}")
    print("This cell runs on the DGX Spark host where training logs are written.")
    print("On Kaggle: not applicable — training runs locally on the DGX Spark.")
else:
    import subprocess
    result = subprocess.run(["tail", "-30", str(log_path)], capture_output=True, text=True)
    print(result.stdout)

    loss_pattern = re.compile(r"'loss': '([0-9.]+)'.*'epoch': '([0-9.]+)'")
    ckpt_pattern = re.compile(r"\[ckpt\] step (\d+)")
    losses, checkpoints = [], []
    with open(log_path) as f:
        for line in f:
            m = loss_pattern.search(line)
            if m:
                losses.append((float(m.group(2)) * MAX_STEPS, float(m.group(1))))
            c = ckpt_pattern.search(line)
            if c:
                checkpoints.append(int(c.group(1)))

    if losses:
        print(f"\nLoss trajectory ({len(losses)} points):")
        for step, loss in losses[-10:]:
            print(f"  step ~{step:4.0f}  loss {loss:.4f}")
    if checkpoints:
        print(f"\nCheckpoints saved at steps: {checkpoints}")
        print(f"Latest rolling checkpoint: output/adapter_{RUN_NAME}_ckpt/")

## Package and submit

Packaging runs on the **host** (not inside Docker). `package_submission.sh` reads from the local `output/` directory directly.

Package from the **rolling checkpoint** (`output/adapter_<RUN_NAME>_ckpt/`) immediately after each checkpoint notification — it is overwritten every 100 steps.

```zsh
# Package from the rolling checkpoint (run on HOST, not inside Docker)
bash scripts/package_submission.sh \
  output/adapter_v9_run13_ckpt \
  /tmp/sub_v9_run13_step500

# Submit to Kaggle
kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f /tmp/sub_v9_run13_step500/submission.zip \
  -m "v0.9 run13 step-500: 878M warmstart, 11762 expert LoRA keys"
```

For v0.12 run14:

```zsh
bash scripts/package_submission.sh \
  output/adapter_v12_spark_ckpt \
  /tmp/sub_v12_step<N>

kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f /tmp/sub_v12_step<N>/submission.zip \
  -m "v0.12 step-<N>: 878M warmstart run13, 25500 rows augmented"
```

In [ ]:
import pathlib

sub_dir = pathlib.Path(f"/tmp/sub_{RUN_NAME}")
if not sub_dir.exists():
    print(f"No submission at {sub_dir}")
    print("Run package_submission.sh on the DGX Spark host after a checkpoint is saved.")
    print("On Kaggle: not applicable — packaging runs on the host, not here.")
else:
    import zipfile, tempfile, os
    from safetensors import safe_open

    zip_path = sub_dir / "submission.zip"
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        st_names = [n for n in names if n.endswith('.safetensors')]
        print(f"submission.zip: {zip_path.stat().st_size / 1e6:.1f} MB")
        print(f"Files in zip: {len(names)}")
        for n in sorted(names):
            info = zf.getinfo(n)
            print(f"  {n}: {info.file_size / 1e6:.1f} MB")

        if st_names:
            with tempfile.TemporaryDirectory() as tmp:
                zf.extract(st_names[0], tmp)
                st_path = os.path.join(tmp, st_names[0])
                with safe_open(st_path, framework='pt') as f:
                    keys = list(f.keys())
            n_expert = sum(1 for k in keys if 'experts.' in k and 'shared_' not in k)
            n_base   = len(keys) - n_expert
            print(f"\nAdapter keys: {len(keys)} total")
            print(f"  Base (attn + shared expert): {n_base}")
            print(f"  Per-expert MoE LoRA:         {n_expert}")
            print(f"  Expected: 186 base + 11776 expert = 11962")

## Adapter key taxonomy — what gets submitted

NemotronH has three categories of LoRA-eligible layers:

| Layer type | Keys | Compatible? | DGX Spark | RTX Pro 6000 |
|---|---|---|---|---|
| Attention (`q/k/v/o_proj`) | 48 | ✓ | ✓ trained | ✓ trained |
| Shared experts (`shared_experts.up/down_proj`) | 92 | ✓ | ✓ trained | ✓ trained |
| Mamba SSM (`in_proj`, stripped `out_proj`) | 92 | ✓ | ✓ (in_proj only) | ✓ (run6+) |
| Routed experts (`experts.{j}.up/down_proj`) | **11,776** | ✓ via PEFT | **✓ 11,776 keys** | ✗ fused format, dropped |

**Key counts by run:**

| Runs | Keys | Trainable params | Kaggle Score cap |
|---|---|---|---|
| Kaggle run1–5 (7 targets) | 140 | 27M | ~0.56 |
| Kaggle run6–7 (9 targets) | 232 | 27M | ~0.57 |
| **DGX Spark run13–run14** | **11,962** | **878M** | **0.58+** |

The routed-expert PEFT keys have the form `base_model.model.model.layers.{i}.mixer.experts.{j}.up_proj.lora_A.default.weight` — 23 layers × 128 experts × 2 proj directions × 2 (lora_A/B) = 11,776 keys.  
`package_submission.sh` converts `expert_lora_weights.pt` into these keys and merges them into the submission safetensors. See `docs/adr/0005-adapter-key-filtering-for-submission.md` for full diagnosis.

## Session-to-session resume guide

### After any checkpoint — package and submit

```zsh
# Package from rolling checkpoint before next checkpoint overwrites it
# Rolling checkpoint is at output/adapter_<RUN_NAME>_ckpt/ (updated every 100 steps)
bash scripts/package_submission.sh \
  output/adapter_<RUN_NAME>_ckpt \
  /tmp/sub_<RUN_NAME>_step<N>
```

### Resume from a specific Trainer checkpoint

Trainer checkpoints (optimizer state + expert LoRA) are saved at `output/adapter_<RUN_NAME>/checkpoint-<step>/`. Pass via `RESUME_FROM_CHECKPOINT`:

```zsh
RESUME_FROM_CHECKPOINT=output/adapter_v9_run13/checkpoint-500 \
bash scripts/run_train_v9.sh
```

### Memory budget (GB10, 130.7 GB HBM)

| Phase | Model BF16 | Expert LoRA | AdamW | Activations (GC) | Peak |
|---|---|---|---|---|---|
| Model load | 60 GB | — | — | — | 60 GB |
| seq=2048 training | 60 GB | ~3 GB | ~7 GB | ~5 GB | ~75 GB ✓ |
| seq=4096 training | 60 GB | ~3 GB | ~7 GB | ~10 GB | ~80 GB ✓ |
| seq=8192 training | 60 GB | ~3 GB | ~7 GB | ~20 GB | ~90 GB (tight) |

Gradient checkpointing is enabled via `_set_gradient_checkpointing()` (NemotronH native — `supports_gradient_checkpointing=False` blocks the standard path).